# LOG05 | 미션 09 | Student Workbook

## 이번 미션에서 하는 일

**피드백 반영 체크리스트 항목 보완 및 실무 적용성 검증**

1. 실습 목적 및 개요 역할 및 배경: 학습자는 무인 보급장비 투입 전 상태·책임·중지조건 체크리스트 교육용 실습의 참여자로서 임무를 수행합니다. 핵심 과제: 앞서 도출된 AAR 피드백을 반영하여 체크리스트 항목을 보완하고, 보완 전후 기준이 동일한 사례에서 일관된 판단을 내릴 수 있는지 실무 적용성을 검증한 뒤, 핵심 결심 사항을 1페이지 분량으로 요약합니다. 2. 주의 및 검증 지침 데이터 해석의 엄정성: 자료나 AI 설명을 철저히 검증하지 않으면 미확인 기록이나 모의 회신을 실제 현장 확인이나 승인으로 오인할 수 있으므로 주의해야 합니다. 수행 방식: 본인 판단, AI 초안 및 역할 대화, 근거 대조 및 보완, 최종 기록의 전 과정을 원본 자료 범위 내에서 엄정하게 수행합니다. 3. 산출물 및 소요 시간 안내 결과 파일 목록: briefing/log05_commander_1page_brief.md LOG05_M09_forms.md forms/LOG05_M09_forms.md 시간 안내: 원본 예상 시간은 3시간이며, CMS 편성은 원본 durationHours 를 올림하여 3시간으로 구성되어 있습니다 (실제 소요 시간은 실습 완료 후 확인).

**학습 목표:** 가상 자료를 읽고 자신의 최초 판단을 적은 다음, AI 초안을 원자료와 대조하여 문서와 검증 근거를 완성합니다. 실제 장비 제어·비행·파괴·무력화 방법을 설계하거나 실행하는 훈련은 아닙니다.

**준비물:** 이 Notebook, 미션 9의 가상 자료 및 빈 작성 양식, 기관이 허용한 AI 환경(가능한 경우).

**학습 환경:** Colab은 인터넷이 필요합니다. 폐쇄망에서는 사전 설치된 Jupyter 또는 별도 Markdown 양식으로 진행할 수 있으며 외부 AI를 사용할 수 없으면 미실시로 기록하세요.

**학습 순서:** 자료 먼저 읽기 → 본인 생각 기록 → AI 질문 → 원문 근거 대조 → 수정 → 저장·제출.

**구분:** 파일 저장 / 실습 수행 / 평가 기준 충족 / 실제 교관 승인은 각각 다른 상태입니다.

## 📦 실습자료 업로드 (모든 미션 공통 · 실습 전 필수)

아래 **「LOG05 실습자료 ZIP 업로드 및 압축 해제」** 코드셀을 **먼저 실행**하세요.

1. 교육기관에서 제공한 **`LOG05_실습프로젝트.zip`**을 준비합니다. 파일명이 다르더라도 LOG05 자료가 담긴 정상적인 ZIP 파일이면 업로드할 수 있습니다.
2. **Google Colab:** 셀 실행 후 나타나는 업로드 창에서 ZIP을 선택하면 자동으로 압축을 해제합니다. **로컬 Jupyter:** ZIP 선택 후 **「선택한 ZIP 압축 해제」** 버튼을 누릅니다.
3. 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

**주의:** 업로드가 끝나지 않았다면 미션 실습을 시작하지 마세요. ZIP을 올렸다는 사실만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.


In [ ]:
# @title 📦 LOG05 실습자료 ZIP 업로드 및 압축 해제 { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

display(Markdown("**실습 시작 전 필수:** 교육기관에서 제공한 `LOG05_실습프로젝트.zip`을 업로드하세요. 이름이 다르더라도 정상적인 `.zip` 파일을 선택할 수 있지만, **LOG05 실습자료가 포함된 ZIP**이어야 합니다. 업로드 및 압축 해제를 완료한 후 다음 실습 단계로 진행하세요."))
try:
    from google.colab import files as _colab_files
    print("📤 업로드 창에서 LOG05 실습자료 ZIP 파일을 선택하세요.")
    _uploaded = _colab_files.upload()
    _zip_files = [(name, content) for name, content in _uploaded.items() if name.lower().endswith('.zip')]
    if not _zip_files:
        print("⚠️ ZIP 파일이 업로드되지 않았습니다. 이 셀을 다시 실행하세요.")
    else:
        if len(_zip_files) > 1:
            print("ℹ️ 여러 ZIP 중 첫 번째 파일을 사용합니다:", _zip_files[0][0])
        _name, _data = _zip_files[0]
        _zip_path = Path.cwd() / Path(_name).name
        _zip_path.write_bytes(_data)
        _log05_extract_zip(_zip_path)
except ImportError:
    import ipywidgets as _widgets
    _file_input = _widgets.FileUpload(accept='.zip', multiple=False, description='ZIP 파일 선택')
    _extract_button = _widgets.Button(description='선택한 ZIP 압축 해제', button_style='success')
    _upload_message = _widgets.Output()
    def _get_uploaded_zip():
        value = _file_input.value
        if not value:
            raise ValueError('먼저 LOG05 실습자료 ZIP을 선택하세요.')
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        name = item['name']
        data = item['content']
        if not name.lower().endswith('.zip'):
            raise ValueError('ZIP 파일만 선택할 수 있습니다.')
        return name, bytes(data)
    def _on_extract(_):
        with _upload_message:
            _upload_message.clear_output()
            try:
                name, data = _get_uploaded_zip()
                target = Path.cwd() / Path(name).name
                target.write_bytes(data)
                _log05_extract_zip(target)
            except Exception as exc:
                print('⚠️ 업로드 또는 압축 해제 실패:', exc)
    _extract_button.on_click(_on_extract)
    display(_widgets.VBox([_file_input, _extract_button, _upload_message]))
    print('로컬 Jupyter: ZIP을 선택한 뒤 「선택한 ZIP 압축 해제」 버튼을 누르세요.')


## 원본 가상 자료와 작성 양식

아래는 첨부된 **LOG05 실습자료 원본**에서 미션별로 추출한 자료입니다. 모의 기록에서 아직 공개되지 않은 카드나 결과는 미리 확정하지 마세요.

미션 9. 피드백 반영 체크리스트 항목 보완 및 실무 적용성 검증

**가. 제공 자료 - 가상 재검증 사례 7건 [교육용가상자료]**

| 사례 | 내용 | 개선 전 판단 | 개선 후 판단(조별 작성) |
| --- | --- | --- | --- |
| V1 | UD-1 배터리 잔량 표시만 확인, 부하 전압 미확인 | 승인 |  |
| V2 | UV-2 링크 품질 합격이나 로그 1회 | 승인 |  |
| V3 | UD-1 적재 한도 도달, 저울 기록 없음 | 승인 |  |
| V4 | 풍속 예보와 측정값 상충 | 승인 |  |
| V5 | 점검표 서명 없이 승인권자 구두 승인 | 승인 |  |
| V6 | 4개 영역 합격, 신뢰도 G2 이상, 서명 완비 | 승인 |  |
| V7 | 중지 수단(통신)이 확보되지 않은 구간 투입 | 승인 |  |

**나. 작성 서식 1 - 개선 전후 기준 비교표**

| 항목 | 개선 전 기준 | 개선 후 기준 | 바뀐 이유 | 근거 미션 |
| --- | --- | --- | --- | --- |
| 배터리 점검 |  |  |  |  |
| 링크 점검 |  |  |  |  |
| 적재 점검 |  |  |  |  |
| 기상 점검 |  |  |  |  |
| 서명·승인 확인 |  |  |  |  |

**나. 작성 서식 2 - 재검증 결과표**

| 사례 | 개선 전 판단 | 개선 후 판단 | 일치 여부 | 소요 시간(분) |
| --- | --- | --- | --- | --- |
| V1 |  |  |  |  |
| V2 |  |  |  |  |
| V3 |  |  |  |  |
| V4 |  |  |  |  |
| V5 |  |  |  |  |
| V6 |  |  |  |  |
| V7 |  |  |  |  |

**나. 작성 서식 3 - 1페이지 브리핑 BLUF 틀**

| 결론(첫 문장) |  |
| --- | --- |

| 핵심 결심 사항 |  |
| --- | --- |

| 위험 요소(신호등 표시: 녹·황·적) |  |
| --- | --- |

| 대안 3종(장단점 한 줄) |  |
| --- | --- |

| 근거(미션·가상 자료) |  |
| --- | --- |

**나. 작성 서식 4 - 브리핑 리허설 기록**

| 항목 | 목표 | 결과 |
| --- | --- | --- |
| 발표 시간 | 3분 이내 |  |
| 분량 | A4 1페이지 |  |
| 질의 응답 | 3건 이상 답변 |  |
| 결론 먼저 제시 | 첫 문장 |  |

**다. 점검표**

| 점검 항목 | 확인 |
| --- | --- |
| 개선 전후 기준의 바뀐 조건과 이유를 모두 적었다 | (   ) |
| 재검증 7건 모두 개선 전후 판단을 비교했다 | (   ) |
| 브리핑이 결론부터 시작하고 A4 1페이지에 들어간다 | (   ) |
| 핵심 결심 사항, 위험 요소, 대안 3종이 모두 있다 | (   ) |
| 3분 안에 발표하고 질의에 답했다 | (   ) |
| 점검에 걸리는 시간을 재서 20분 이내 완료 가능한지 확인했다 | (   ) |
| 기준 변경으로 생긴 판단 차이를 체크리스트 항목과 연결했다 | (   ) |

## 단계별 수행 안내와 미션 전용 AI 활용

### 활동 1. 전후 기준과 출처를 고정한다

**무엇을 하나요?** 1. 단계 개요 목표: 미션 8까지 수행한 피드백과 개선 사항을 바탕으로, 후속 실무 적용성 검증의 대상이 되는 V1부터 V7까지의 원본 이전 판정과 적용 기준 을 명확히 고정합니다. 수행 방식: 주관적이고 임의적인 해석을 방지하기 위해 각 버전별 판정의 출처와 미확인 영역, 그리고 소요 시간 측정 방식을 사전에 엄정하게 정의하여 평가의 객관적 기준선을 확립합니다. 2. 세부 수행 절차 V1～V7 원본 이전 판정 및 적용 기준 정의: 각 버전별(V1～V7) 무인 보급장비 운용 및 중지 조건 판정의 원본 내용과, 피드백 반영 전후의 판단 기준 변화 요소를 명확히 분리하여 기록합니다. 미확인 영역 및 정보 공백 식별: 가상 자료 및 로그 속에서 실제 현장 확인이 누락되었거나 통신 제약으로 인해 검증되지 못한 '미확인' 상태의 범위를 사전에 규정합니다. 시간 측정 방법 표준화: 후속 단계(2단계)에서 진행될 문서 점검 및 재검증 과정의 소요 시간을 객관적으로 측정하기 위한 기준과 방식을 고정합니다. 3. 주의 및 검증 사항 이전 단계의 판정 기준을 임의로 왜곡하거나 누락된 항목을 사전에 모두 완료된 것처럼 설정하지 않도록 주의해야 합니다. 본 단계에서 고정된 기준과 출처 정보는 후속 2단계(직접 재검증) 및 3단계(1페이지 브리핑 작성)의 핵심 뼈대가 되므로 문서에 명확히 명시해야 합니다.

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 2. 직접 재검증하고 시간을 잰다

**무엇을 하나요?** 1. 단계 개요 목표: 피드백 반영 전후의 기준이 동일한 7건의 대표 사례를 대상으로, 보완된 체크리스트와 중지 조건을 적용하여 직접 재검증 을 수행하고 각 과정별 문서 점검 소요 시간을 정확하게 측정·기록합니다. 수행 방식: 주관적 추정이나 막연한 비교를 배제하고, 7가지 개별 시나리오별로 전후 판정의 변화 여부와 복수의 검토자 간 일치율을 정밀하게 대조하며 실무 적용 타당성을 객관적으로 증명합니다. 2. 세부 수행 절차 7건의 전후 판단 및 변화 양상 대조: 동일한 상황 조건에서 피드백 반영 이전의 판정과 반영 이후의 판정이 어떻게 달라졌는지 7건의 사례별로 항목화하여 비교합니다. 검토자 간 일치율 확인: 보완된 기준에 따라 판단을 수행할 때 복수의 검토자(또는 평가자) 간에 결론이 일치하는지 교차 검증하고 불일치 원인을 분석합니다. 문서 점검 및 재검증 소요 시간 측정: 7건의 사례를 검토하고 판정을 내리는 데 소요되는 실제 시간을 정확히 측정하여 기록함으로써, 현장 적용 시 소요되는 실무적 타당성과 효율성을 검증합니다. 3. 주의 및 검증 사항 단순히 결과가 맞았다고 넘어가거나 시간을 임의로 추정하여 적지 않도록, 실제 검증 과정에서 발생한 시간과 오차를 엄정하게 기록해야 합니다. 본 단계에서 도출된 7건의 재검증 결과와 실측 시간 데이터는 후속 미션(3단계 브리핑 작성 및 4단계 최종 이관)의 핵심 증거 자료로 활용됩니다.

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 3. 1페이지·발표·질의를 확인한다

**무엇을 하나요?** 1. 단계 개요 목표: 앞선 단계를 통해 검증된 무인 보급장비 운용 개선 결과를 바탕으로 지휘관 보고용 1페이지 브리핑(Briefing)을 작성하고, 핵심 결론(BLUF), 핵심 결심, 상태 신호등, 대안 3종을 명확히 구성한 뒤 실제 3분 발표 및 3건의 질의응답 과 인쇄 분량을 점검합니다. 수행 방식: 방대한 분석 내용을 지휘관이 단 시간에 직관적으로 파악할 수 있도록 핵심 정보 위주로 압축하고, 실무 발표 및 방어 과정을 통해 브리핑 문서의 완성도와 실효성을 최종 검증합니다. 2. 세부 수행 절차 1페이지 브리핑 핵심 요소 구성: BLUF(Bottom Line Up Front): 결론과 핵심 제안을 문서 최상단에 간결하게 요약 명시. 핵심 결심: 지휘관이 즉각적으로 내릴 수 있도록 의사결정 필요 사안을 명확히 제시. 상태 신호등: 장비 상태 및 리스크 지표를 직관적인 신호 체계(녹색·황색·적색 등)로 시각화. 대안 3종 비교: 상황별로 적용 가능한 3가지 대응 대안의 장단점 정리. 3분 발표 및 3건의 질의응답 실습: 작성된 1페이지 브리핑을 바탕으로 3분 이내의 압축적인 구두 발표를 수행하고, 예상되는 쟁점에 대한 3가지 질의(Critical Questions)와 방어 논리를 점검합니다. 문서 레이아웃 및 인쇄 분량 확인: 실제 출력 또는 디지털 열람 시 페이지 넘침 없이 정확히 1페이지 분량에 모든 핵심 정보가 밀도 있게 담겼는지 레이아웃을 검토합니다. 3. 주의 및 검증 사항 분량을 맞추기 위해 필수적인 근거 데이터나 핵심 결심 사항을 누락하지 않도록 주의해야 합니다. 본 단계를 통해 완성된 내용은 최종 브리핑 결과 파일( briefing/log05_commander_1page_brie

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 4. 개선 결과와 한계를 넘긴다

**무엇을 하나요?** 1. 단계 개요 목표: 앞선 1～3단계를 통해 도출된 전후 판단 기준 데이터, 7건의 재검증 결과, 실측 시간 및 회고 내용, 20분 분석 결과, 그리고 지휘관 1페이지 브리핑 증거 를 체계적으로 정리하여 후속 최종 미션인 미션 10(무인 보급장비 투입 전 상태·책임·중지조건 표준 체크리스트 완성)으로 정확하고 누락 없이 인계합니다. 수행 방식: 단편적인 실습 결과로 끝나지 않도록, 실무 검증 과정에서 확인된 명확한 개선 성과와 잔존하는 한계점(정보 공백, 제약 조건 등)을 모두 포함하여 후속 단계의 기초 자료로 이관합니다. 2. 세부 수행 절차 전후 기준 및 7건 재검증 결과 이관: 피드백 반영 전후의 판단 기준 변화 내역과 7건의 대표 사례별 검증 결과 및 검토자 간 일치율 데이터를 표준 양식에 맞춰 정돈합니다. 실측 시간 및 회고(Retrospective) 기록 정리: 각 문서 점검 및 재검증 과정에서 측정된 실제 소요 시간과, 실무 적용 과정에서 도출된 한계점 및 소회를 빠짐없이 기록합니다. 20분 분석 결과 및 브리핑 증거 취합: 압축 분석 결과와 지휘관 보고용 1페이지 브리핑( briefing/log05_commander_1page_brief.md ) 및 관련 양식 파일( LOG05_M09_forms.md , forms/LOG05_M09_forms.md )의 검증 증거를 최종 편성화합니다. 미션 10과의 연계 및 최종 이관: 정리된 모든 핵심 증거와 결과물을 후속 미션 10의 표준 체크리스트 완성 작업에 즉시 활용할 수 있도록 정확한 경로와 포맷으로 넘깁니다. 3. 주의 및 검증 사항 실제 검증이나 실측이 이루어지지 않은 항목을 임의로 완료된 것처럼 작성하여 인계하지 않도록 주의해야 합

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

## 제출 전에 확인할 기준

**평가 항목 1:** [정량적 평가] [필수] 체크리스트 보완·1페이지 브리핑 — 핵심 결심 사항, 위험 요소, 대안 3종이 1페이지(A4) 안에 완결되는가, 재검증 7건의 개선 전후 비교와 적용성 검증(점검 20분 이내 여부)을 마쳤는가, BLUF 준수(결론 먼저, 3분 이내 발표)와 신호등 표시가 있는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영. PASS 증거 AAR 피드백을 반영해 체크리스트 항목을 보완하고, 보완 전후 기준이 같은 사례에서 일관된 판단을 내는지 실무 적용성을 검증한 뒤 핵심 결심 사항을 1페이지로 요약한다.의판단이제공자료·역할책임·교육용가정과연결됩니다. 사실/제안/미확인과실제/모의수행을구분합니다. 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다. 미실시를 PASS로 쓰지 않는다. 파일: briefing/log05_commander_1page_brief.md, LOG05_M09_forms.md, forms/LOG05_M09_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다. 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다. 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**중요:** 정답·실제 측정값·사람의 서명이나 승인을 AI가 만들어 주지 않도록 합니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m09", "steps": [{"title": "전후 기준과 출처를 고정한다"}, {"title": "직접 재검증하고 시간을 잰다"}, {"title": "1페이지·발표·질의를 확인한다"}, {"title": "개선 결과와 한계를 넘긴다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("ipywidgets가 필요합니다. 폐쇄망에서는 사전 설치된 로컬 Jupyter 또는 제공된 Markdown 양식을 사용하세요.") from exc

MISSION_DATA = json.loads(MISSION_JSON)
CODE = MISSION_DATA['code']
OUT_DIR = Path(CODE + '_results')
OUT_DIR.mkdir(exist_ok=True)
BACKUP = OUT_DIR / (CODE + '_record.json')
fields = {}
for no, item in enumerate(MISSION_DATA['steps'], 1):
    fields['activity_%02d' % no] = widgets.Textarea(
        value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
        description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
        style={'description_width':'75px'})
for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                   ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                   ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
    fields[key] = widgets.Textarea(value='', description=label,
       layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

def payload():
    return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'answers':{key:w.value for key,w in fields.items()},
            'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

def save_record(_=None):
    obj=payload()
    if BACKUP.exists():
        stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
    BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
    status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

def export_json(_=None):
    save_record()
    raw=BACKUP.read_bytes()
    try:
        from google.colab import files
        files.download(str(BACKUP))
    except ImportError:
        print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

def restore_record(_=None):
    try:
        obj=json.loads(restore_input.value)
        if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
        save_record()  # 복원 전 백업 저장
        for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
        status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
    except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

def export_review_zip(_=None):
    save_record()
    target=OUT_DIR / (CODE+'_review.zip')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
        zz.write(BACKUP,arcname=BACKUP.name)
        text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
        zz.writestr(CODE+'_work.md',text)
        zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
    try:
        from google.colab import files
        files.download(str(target))
    except ImportError:
        print('Colab이 아닙니다. ZIP 위치:',target.resolve())

for i,item in enumerate(MISSION_DATA['steps'],1):
    display(Markdown('### 활동 %d. %s' % (i,item['title'])))
    display(fields['activity_%02d' % i])
for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
 ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
    display(Markdown('### '+label));display(fields[key])
buttons=[]
for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
    b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)


## 마무리

작성 JSON과 검토용 ZIP을 내려받아 보관하세요. ZIP은 **검토용**이며 평가 PASS나 교관 승인 증명이 아닙니다. 전 단계 결과가 필요하면 본인의 실제 이전 기록을 사용하고, 기록이 없으면 **미실시**로 남기세요.